# Book Recommendation System using a Medallion Data Lake

**Data Mining Lab Assignment**

LCI023008 - SHREYA JAMBHALE

In [1]:
import os
import zipfile
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors
from IPython.display import display

for folder in ["bronze", "silver", "gold"]:
    os.makedirs(folder, exist_ok=True)

print("Folders created.")

Folders created.


## 1. Get the dataset


In [2]:
from google.colab import files

uploaded = files.upload()
zip_path = next(iter(uploaded.keys()))

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("raw_data")

print("Extracted files:", os.listdir("raw_data"))

Saving book-crossings.zip to book-crossings.zip
Extracted files: ['BX-Book-Ratings.csv', 'BX-Books.csv', 'BX-Users.csv']


## 2. Bronze layer — raw data

We read the three CSV files with their original text values, add an ingestion timestamp, and save them as Parquet. No cleaning is done in Bronze.

In [5]:
users_raw = pd.read_csv(
    "raw_data/BX-Users.csv", sep=";", encoding="latin-1",
    dtype=str, keep_default_na=False
)
books_raw = pd.read_csv(
    "raw_data/BX-Books.csv", sep=";", encoding="latin-1",
    dtype=str, keep_default_na=False,
    escapechar="\\"
)
ratings_raw = pd.read_csv(
    "raw_data/BX-Book-Ratings.csv", sep=";", encoding="latin-1",
    dtype=str, keep_default_na=False
)

ingestion_time = datetime.now(timezone.utc).isoformat()

for df in [users_raw, books_raw, ratings_raw]:
    df["ingestion_timestamp"] = ingestion_time

users_raw.to_parquet("bronze/users.parquet", index=False)
books_raw.to_parquet("bronze/books.parquet", index=False)
ratings_raw.to_parquet("bronze/ratings.parquet", index=False)

print("Bronze files saved.")
print("Users:", users_raw.shape)
print("Books:", books_raw.shape)
print("Ratings:", ratings_raw.shape)

Bronze files saved.
Users: (278858, 4)
Books: (271379, 9)
Ratings: (1149780, 4)


## 3. Explore the Bronze data (EDA)

The next cell prints the shape, data types, missing values, rating distribution, age/year ranges, ratings per user/book, and matrix sparsity. Since the raw files are kept as strings, the numeric columns are converted temporarily only for EDA.

In [7]:
users_b = pd.read_parquet("bronze/users.parquet")
books_b = pd.read_parquet("bronze/books.parquet")
ratings_b = pd.read_parquet("bronze/ratings.parquet")

def basic_profile(name, df):
    print(f"\\n--- {name} ---")
    print("Shape:", df.shape)
    print("Data types:")
    print(df.dtypes)
    print("Missing values:")
    print(df.isna().sum())

basic_profile("Users", users_b)
basic_profile("Books", books_b)
basic_profile("Ratings", ratings_b)

ratings_num = pd.to_numeric(ratings_b["Book-Rating"], errors="coerce")
ages_num = pd.to_numeric(users_b["Age"].replace("NULL", np.nan), errors="coerce")
years_num = pd.to_numeric(books_b["Year-Of-Publication"].replace("NULL", np.nan), errors="coerce")

print("\\nRating distribution:")
print(ratings_num.value_counts(dropna=False).sort_index())
print("Number of zero ratings:", int((ratings_num == 0).sum()))

print("\\nUser age summary:")
print(ages_num.describe())
print("Ages below 5:", int((ages_num < 5).sum()))
print("Ages above 100:", int((ages_num > 100).sum()))

current_year = datetime.now().year
print("\\nPublication year summary:")
print(years_num.describe())
print("Year 0 count:", int((years_num == 0).sum()))
print("Future publication year count:", int((years_num > current_year).sum()))

ratings_per_user = ratings_b.groupby("User-ID").size()
ratings_per_book = ratings_b.groupby("ISBN").size()
print("\\nRatings per user summary:")
print(ratings_per_user.describe())
print("Users with fewer than 10 ratings:", int((ratings_per_user < 10).sum()))
print("\\nRatings per book summary:")
print(ratings_per_book.describe())
print("Books with fewer than 10 ratings:", int((ratings_per_book < 10).sum()))

possible_pairs = ratings_b["User-ID"].nunique() * ratings_b["ISBN"].nunique()
actual_pairs = ratings_b[["User-ID", "ISBN"]].drop_duplicates().shape[0]
sparsity = 100 * (1 - actual_pairs / possible_pairs)
print(f"\\nMatrix sparsity: {sparsity:.4f}%")

\n--- Users ---
Shape: (278858, 4)
Data types:
User-ID                object
Location               object
Age                    object
ingestion_timestamp    object
dtype: object
Missing values:
User-ID                0
Location               0
Age                    0
ingestion_timestamp    0
dtype: int64
\n--- Books ---
Shape: (271379, 9)
Data types:
ISBN                   object
Book-Title             object
Book-Author            object
Year-Of-Publication    object
Publisher              object
Image-URL-S            object
Image-URL-M            object
Image-URL-L            object
ingestion_timestamp    object
dtype: object
Missing values:
ISBN                   0
Book-Title             0
Book-Author            0
Year-Of-Publication    0
Publisher              0
Image-URL-S            0
Image-URL-M            0
Image-URL-L            0
ingestion_timestamp    0
dtype: int64
\n--- Ratings ---
Shape: (1149780, 4)
Data types:
User-ID                object
ISBN                   ob

### EDA Observations

- **Shape & Missing Values:** Users: 278,858 rows, Books: 271,379 rows, Ratings: 1,149,780 rows. No missing values detected. All columns are initially read as strings.

- **Rating Distribution:** 716,109 ratings are 0 (no explicit rating). These will be removed, keeping ratings from 1–10.

- **Age & Publication Year:** 882 users are below age 5 and 366 are above 100. There are 4,619 books with publication year 0 and 11 with future years. These invalid values will be replaced with null.

- **Ratings per User/Book:** 92,186 users and 322,237 books have fewer than 10 ratings. I will keep only users and books with at least 10 ratings.

- **Matrix Sparsity:** The matrix is 99.9968% sparse. A sparse matrix will be used in Gold to save memory.

## 4. Silver layer — clean the data

Chosen threshold: **at least 10 ratings per user and at least 10 ratings per book**. This removes very lightly rated users/books, which provide little information for collaborative filtering. We repeat the filtering until both conditions are satisfied after filtering.

In [8]:
users = pd.read_parquet("bronze/users.parquet").drop(columns=["ingestion_timestamp"])
books = pd.read_parquet("bronze/books.parquet").drop(columns=["ingestion_timestamp"])
ratings = pd.read_parquet("bronze/ratings.parquet").drop(columns=["ingestion_timestamp"])

users = users.drop_duplicates()
books = books.drop_duplicates()
ratings = ratings.drop_duplicates()

users["User-ID"] = users["User-ID"].astype(str).str.strip()
users["Location"] = users["Location"].astype(str).str.strip().str.lower()
users["Age"] = pd.to_numeric(users["Age"].replace("NULL", np.nan), errors="coerce")
users.loc[(users["Age"] < 5) | (users["Age"] > 100), "Age"] = np.nan
median_age = users["Age"].median()
users["Age"] = users["Age"].fillna(median_age)

location_parts = users["Location"].str.split(",", n=2, expand=True)
for i in range(3):
    if i not in location_parts.columns:
        location_parts[i] = ""
users["City"] = location_parts[0].fillna("").str.strip().str.title()
users["State"] = location_parts[1].fillna("").str.strip().str.title()
users["Country"] = location_parts[2].fillna("").str.strip().str.title()
users = users.drop(columns=["Location"])

for col in ["ISBN", "Book-Title", "Book-Author", "Publisher"]:
    books[col] = books[col].astype(str).str.strip()

books["Book-Title"] = books["Book-Title"].str.title()
books["Book-Author"] = books["Book-Author"].str.title()
books["Publisher"] = books["Publisher"].str.title()

books["Year-Of-Publication"] = pd.to_numeric(
    books["Year-Of-Publication"].replace("NULL", np.nan), errors="coerce"
)
current_year = datetime.now().year
invalid_year = (books["Year-Of-Publication"] == 0) | (books["Year-Of-Publication"] > current_year)
books.loc[invalid_year, "Year-Of-Publication"] = np.nan

image_cols = [c for c in books.columns if c.startswith("Image-URL")]
books = books.drop(columns=image_cols)

ratings["User-ID"] = ratings["User-ID"].astype(str).str.strip()
ratings["ISBN"] = ratings["ISBN"].astype(str).str.strip()
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce")
ratings = ratings[ratings["Book-Rating"].between(1, 10)]

ratings = ratings[
    ratings["User-ID"].isin(users["User-ID"]) &
    ratings["ISBN"].isin(books["ISBN"])
].copy()

min_ratings = 10
while True:
    old_count = len(ratings)
    user_counts = ratings["User-ID"].value_counts()
    book_counts = ratings["ISBN"].value_counts()

    valid_users = user_counts[user_counts >= min_ratings].index
    valid_books = book_counts[book_counts >= min_ratings].index

    ratings = ratings[
        ratings["User-ID"].isin(valid_users) &
        ratings["ISBN"].isin(valid_books)
    ].copy()

    if len(ratings) == old_count:
        break

users = users[users["User-ID"].isin(ratings["User-ID"].unique())].copy()
books = books[books["ISBN"].isin(ratings["ISBN"].unique())].copy()

users.to_parquet("silver/users.parquet", index=False)
books.to_parquet("silver/books.parquet", index=False)
ratings.to_parquet("silver/ratings.parquet", index=False)

print("Silver layer saved.")
print("Users:", users.shape)
print("Books:", books.shape)
print("Ratings:", ratings.shape)
print("Minimum user ratings:", ratings.groupby("User-ID").size().min())
print("Minimum book ratings:", ratings.groupby("ISBN").size().min())

Silver layer saved.
Users: (1820, 5)
Books: (2030, 5)
Ratings: (41456, 3)
Minimum user ratings: 10
Minimum book ratings: 10


## 5. Gold layer — model-ready data

The matrix is stored with **books as rows and users as columns**. This orientation is useful for comparing books by their rating patterns across users. We also save the ID lists so the sparse matrix rows/columns can be mapped back to ISBNs and user IDs.

In [9]:
book_ids = sorted(ratings["ISBN"].unique())
user_ids = sorted(ratings["User-ID"].unique())

book_to_row = {book_id: i for i, book_id in enumerate(book_ids)}
user_to_col = {user_id: i for i, user_id in enumerate(user_ids)}

row_indices = ratings["ISBN"].map(book_to_row).to_numpy()
col_indices = ratings["User-ID"].map(user_to_col).to_numpy()
values = ratings["Book-Rating"].astype(float).to_numpy()

rating_matrix = csr_matrix(
    (values, (row_indices, col_indices)),
    shape=(len(book_ids), len(user_ids))
)

save_npz("gold/book_user_matrix.npz", rating_matrix)
pd.DataFrame({"ISBN": book_ids}).to_csv("gold/book_ids.csv", index=False)
pd.DataFrame({"User-ID": user_ids}).to_csv("gold/user_ids.csv", index=False)

book_lookup = books[["ISBN", "Book-Title", "Book-Author"]].drop_duplicates("ISBN")
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

user_aggregates = ratings.groupby("User-ID")["Book-Rating"].agg(
    average_rating="mean", number_of_ratings="count"
).reset_index()

book_aggregates = ratings.groupby("ISBN")["Book-Rating"].agg(
    average_rating="mean", number_of_ratings="count"
).reset_index()

user_aggregates.to_parquet("gold/user_aggregates.parquet", index=False)
book_aggregates.to_parquet("gold/book_aggregates.parquet", index=False)

print("Gold layer saved.")
print("Matrix shape (books x users):", rating_matrix.shape)
print("Number of ratings in matrix:", rating_matrix.nnz)

Gold layer saved.
Matrix shape (books x users): (2030, 1820)
Number of ratings in matrix: 41456


## 6. Build the item-based collaborative filtering recommender

Cosine similarity compares books using the users who rated them. For a user, we take their highly rated books (rating 7 or above), find similar books, combine similarity scores weighted by the user's rating, and remove books they have already rated.

In [10]:
rating_matrix = load_npz("gold/book_user_matrix.npz").tocsr()
book_ids = pd.read_csv("gold/book_ids.csv")["ISBN"].astype(str).tolist()
user_ids = pd.read_csv("gold/user_ids.csv")["User-ID"].astype(str).tolist()
book_lookup = pd.read_parquet("gold/book_lookup.parquet")

book_to_row = {book_id: i for i, book_id in enumerate(book_ids)}
user_to_col = {user_id: i for i, user_id in enumerate(user_ids)}

neighbors_model = NearestNeighbors(metric="cosine", algorithm="brute")
neighbors_model.fit(rating_matrix)

def similar_books(book_title, n=10):
    """Return the n most similar books to the given title."""
    matches = book_lookup[
        book_lookup["Book-Title"].str.casefold() == book_title.strip().casefold()
    ]

    if matches.empty:
        matches = book_lookup[
            book_lookup["Book-Title"].str.contains(book_title, case=False, na=False)
        ]

    if matches.empty:
        print("Book title not found.")
        return pd.DataFrame(columns=["Book-Title", "Book-Author", "Similarity"])

    isbn = matches.iloc[0]["ISBN"]
    row = book_to_row[isbn]

    distances, indices = neighbors_model.kneighbors(
        rating_matrix[row], n_neighbors=min(n + 1, len(book_ids))
    )

    results = []
    for distance, idx in zip(distances[0], indices[0]):
        if idx == row:
            continue
        results.append({
            "ISBN": book_ids[idx],
            "Similarity": 1 - float(distance)
        })

    result_df = pd.DataFrame(results)
    result_df = result_df.merge(book_lookup, on="ISBN", how="left")
    return result_df[["Book-Title", "Book-Author", "Similarity"]].head(n)

def recommend(user_id, n=10):
    """Recommend books for a user based on their highly rated books."""
    user_id = str(user_id)
    if user_id not in user_to_col:
        print("User not found in the cleaned dataset.")
        return pd.DataFrame(columns=["Book-Title", "Book-Author", "Score"])

    user_col = user_to_col[user_id]
    user_ratings = rating_matrix[:, user_col].tocoo()

    already_rated_rows = set(user_ratings.row.tolist())

    liked_rows = [
        (row, rating)
        for row, rating in zip(user_ratings.row, user_ratings.data)
        if rating >= 7
    ]

    if not liked_rows:
        print("This user has no ratings of 7 or above.")
        return pd.DataFrame(columns=["Book-Title", "Book-Author", "Score"])

    scores = {}
    weights = {}

    for liked_row, user_rating in liked_rows:
        distances, indices = neighbors_model.kneighbors(
            rating_matrix[liked_row],
            n_neighbors=min(21, len(book_ids))
        )

        for distance, candidate_row in zip(distances[0], indices[0]):
            similarity = 1 - float(distance)

            if candidate_row == liked_row or candidate_row in already_rated_rows:
                continue
            if similarity <= 0:
                continue

            scores[candidate_row] = scores.get(candidate_row, 0.0) + similarity * float(user_rating)
            weights[candidate_row] = weights.get(candidate_row, 0.0) + similarity

    if not scores:
        return pd.DataFrame(columns=["Book-Title", "Book-Author", "Score"])

    ranked = sorted(
        scores,
        key=lambda row: scores[row] / weights[row],
        reverse=True
    )[:n]

    output = pd.DataFrame({
        "ISBN": [book_ids[row] for row in ranked],
        "Score": [scores[row] / weights[row] for row in ranked]
    })
    output = output.merge(book_lookup, on="ISBN", how="left")
    return output[["Book-Title", "Book-Author", "Score"]]

## 7. Demo and sanity check

This cell selects three users who have at least one rating of 7 or above, prints the books they rated highly, and then displays recommendations for each.

In [11]:
user_rating_counts = ratings[ratings["Book-Rating"] >= 7].groupby("User-ID").size()
demo_users = user_rating_counts.sort_values(ascending=False).head(3).index.tolist()

for uid in demo_users:
    print("\\n" + "=" * 70)
    print("USER:", uid)

    liked = ratings[
        (ratings["User-ID"] == uid) &
        (ratings["Book-Rating"] >= 7)
    ].merge(book_lookup, on="ISBN", how="left")

    print("\\nBooks this user rated 7 or above:")
    display(liked[["Book-Title", "Book-Author", "Book-Rating"]].head(10))

    print("Recommendations:")
    display(recommend(uid, n=10))

\n======================================================================
USER: 11676
\nBooks this user rated 7 or above:


,Book-Title,Book-Author,Book-Rating
0,Angels,Marian Keyes,8
1,Boy Meets Girl,Meg Cabot,7
2,Watermelon,Marian Keyes,7
3,"Men Are From Mars, Women Are From Venus: A Pra...",John Gray,9
4,The Poisonwood Bible,Barbara Kingsolver,10
5,Daughter Of Fortune : A Novel (Oprah'S Book Cl...,Isabel Allende,10
6,Prodigal Summer,Barbara Kingsolver,10
7,Where The Sidewalk Ends : Poems And Drawings,Shel Silverstein,8
8,A Light In The Attic,Shel Silverstein,7
9,I Know This Much Is True (Oprah'S Book Club),Wally Lamb,10


Recommendations:


,Book-Title,Book-Author,Score
0,James And The Giant Peach,Roald Dahl,10.0
1,The Giving Tree,Shel Silverstein,10.0
2,Great Gatsby (Reissue),F. Scott Fitzgerald,10.0
3,The Hostile Hospital (A Series Of Unfortunate ...,Lemony Snicket,10.0
4,"Eragon (Inheritance, Book 1)",Christopher Paolini,10.0
5,The Voyage Of The Dawn Treader (Rack) (Narnia),C. S. Lewis,10.0
6,The Silver Chair,C. S. Lewis,10.0
7,Prince Caspian (Rack) : The Return To Narnia (...,C. S. Lewis,10.0
8,The Horse And His Boy,C. S. Lewis,10.0
9,The View From Saturday (Jean Karl Books (Paper...,E. L. Konigsburg,10.0


\n======================================================================
USER: 16795
\nBooks this user rated 7 or above:


,Book-Title,Book-Author,Book-Rating
0,Dr. Atkins' New Diet Revolution,Robert C. Atkins,8
1,Patron Saint Of Liars : A Novel,Ann Patchett,8
2,Little Altars Everywhere: A Novel,Rebecca Wells,7
3,One For The Money (Stephanie Plum Novels (Pape...,Janet Evanovich,9
4,The Alchemist: A Fable About Following Your Dream,Paulo Coelho,8
5,The Secret Garden,Frances Hodgson Burnett,8
6,The Bad Beginning (A Series Of Unfortunate Eve...,Lemony Snicket,8
7,The Reptile Room (A Series Of Unfortunate Even...,Lemony Snicket,7
8,Fried Green Tomatoes At The Whistle Stop Cafe,Fannie Flagg,9
9,The Liar'S Club: A Memoir,Mary Karr,10


... (2 outputs left out)

\n======================================================================
USER: 104636
\nBooks this user rated 7 or above:


,Book-Title,Book-Author,Book-Rating
0,Where The Sidewalk Ends : Poems And Drawings,Shel Silverstein,10
1,One Hundred Years Of Solitude (Oprah'S Book Club),Gabriel Garcia Marquez,8
2,One For The Money (Stephanie Plum Novels (Pape...,Janet Evanovich,9
3,Little House In The Big Woods,Laura Ingalls Wilder,9
4,Little House On The Prairie,Laura Ingalls Wilder,10
5,Farmer Boy (Little House),Laura Ingalls Wilder,8
6,On The Banks Of Plum Creek,Laura Ingalls Wilder,9
7,By The Shores Of Silver Lake (Little House),Laura Ingalls Wilder,9
8,These Happy Golden Years (Little House),Laura Ingalls Wilder,9
9,Charlotte'S Web (Trophy Newbery),E. B. White,10


Recommendations:


,Book-Title,Book-Author,Score
0,"Bless Me, Ultima",Rudolfo Anaya,10.0
1,The Voyage Of The Dawn Treader (Rack) (Narnia),C. S. Lewis,10.0
2,Harry Potter And The Order Of The Phoenix (Boo...,J. K. Rowling,10.0
3,A Light In The Attic,Shel Silverstein,10.0
4,Charlie And The Chocolate Factory (Puffin Novels),Roald Dahl,10.0
5,It Came From The Far Side,Gary Larson,10.0
6,Matilda,Roald Dahl,10.0
7,How The Garcia Girls Lost Their Accents (Plume...,Julia Alvarez,10.0
8,One Hundred Years Of Solitude,Gabriel Garcia Marquez,10.0
9,Falling Up,Shel Silverstein,10.0


## 8. DEMO

In [12]:
similar_books("Harry Potter", n=10)

,Book-Title,Book-Author,Similarity
0,Harry Potter And The Chamber Of Secrets (Book 2),J. K. Rowling,0.411046
1,Harry Potter And The Prisoner Of Azkaban (Book 3),J. K. Rowling,0.315639
2,Harry Potter And The Goblet Of Fire (Book 4),J. K. Rowling,0.254112
3,Harry Potter And The Goblet Of Fire (Book 4),J. K. Rowling,0.253747
4,Harry Potter And The Order Of The Phoenix (Boo...,J. K. Rowling,0.244261
5,Harry Potter And The Prisoner Of Azkaban (Book 3),J. K. Rowling,0.203506
6,Harry Potter And The Chamber Of Secrets (Book 2),J. K. Rowling,0.188742
7,Fried Green Tomatoes At The Whistle Stop Cafe,Fannie Flagg,0.188543
8,Love Story,Erich Segal,0.169648
9,Anne Of Green Gables (Anne Of Green Gables Nov...,L.M. Montgomery,0.163291


In [13]:
recommend(demo_users[0], n=10)

,Book-Title,Book-Author,Score
0,James And The Giant Peach,Roald Dahl,10.0
1,The Giving Tree,Shel Silverstein,10.0
2,Great Gatsby (Reissue),F. Scott Fitzgerald,10.0
3,The Hostile Hospital (A Series Of Unfortunate ...,Lemony Snicket,10.0
4,"Eragon (Inheritance, Book 1)",Christopher Paolini,10.0
5,The Voyage Of The Dawn Treader (Rack) (Narnia),C. S. Lewis,10.0
6,The Silver Chair,C. S. Lewis,10.0
7,Prince Caspian (Rack) : The Return To Narnia (...,C. S. Lewis,10.0
8,The Horse And His Boy,C. S. Lewis,10.0
9,The View From Saturday (Jean Karl Books (Paper...,E. L. Konigsburg,10.0


## 9. Final checks


In [14]:
for folder in ["bronze", "silver", "gold"]:
    print(f"\\n{folder}/")
    for filename in sorted(os.listdir(folder)):
        print(" -", filename)

\nbronze/
 - books.parquet
 - ratings.parquet
 - users.parquet
\nsilver/
 - books.parquet
 - ratings.parquet
 - users.parquet
\ngold/
 - book_aggregates.parquet
 - book_ids.csv
 - book_lookup.parquet
 - book_user_matrix.npz
 - user_aggregates.parquet
 - user_ids.csv
